# talkinghead — Kaggle render

This notebook is the runtime for the whole pipeline. Nothing needs to be
installed on your own machine: Kaggle sessions ship with **FFmpeg already on
PATH**, and provide a free GPU for the lipsync stage.

## Before you run

1. Upload `base_loop.mp4` and `reference.wav` as a **private** Kaggle Dataset
   named `talkinghead-assets`. Private matters — it is your face and your
   voice, and a public dataset makes both freely downloadable.
2. Attach it here via **+ Add Input**.
3. For lipsync, set **Settings → Accelerator → GPU**. For audio-only work,
   leave it on CPU: **CPU sessions are unmetered**, so assembly costs nothing
   against your 30 GPU-hours/week.

## Quota strategy

| Stage | Session | Cost |
|---|---|---|
| script prep, TTS, audio assembly, encode | CPU | free, unmetered |
| lipsync | GPU | draws on 30 hr/week |

Iterate on the voice in a CPU session; switch to GPU only when the narration
is final.

## 1. Confirm the host provides what we need

In [ ]:
!ffmpeg -version | head -1
!ffprobe -version | head -1
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'CPU session (fine for audio stages)'

## 2. Install the package

Cloned fresh each session — Kaggle sessions are ephemeral, which is why the
code lives in git rather than in the notebook.

In [ ]:
REPO = "https://github.com/Deveshkumar742/talkinghead.git"

import os, subprocess, sys

if not os.path.exists("/kaggle/working/talkinghead"):
    subprocess.run(["git", "clone", "--depth", "1", REPO,
                    "/kaggle/working/talkinghead"], check=True)

os.chdir("/kaggle/working/talkinghead")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
print("installed")

## 3. Verify the environment and the mounted assets

`host` reports what the session provides; `check` validates the recordings
against the active profile — including whether the base loop's resolution
matches what you are trying to render.

In [ ]:
!talkinghead host
print("-" * 60)
!talkinghead check || true

## 4. Write your script

This text is spoken verbatim — the pipeline does not rewrite or polish it.

In [ ]:
from pathlib import Path

SCRIPT = """\
Hi, I'm Devesh. This is a test of the pipeline end to end.

I'm checking three things: whether the mouth tracks the words, whether the
teeth hold up at full resolution, and whether the voice sounds like me.
"""

script_path = Path("/kaggle/working/script.txt")
script_path.write_text(SCRIPT, encoding="utf-8")

!talkinghead prep /kaggle/working/script.txt

## 5. Render

Blocked until Phases 2 and 3 land (the Chatterbox and LatentSync providers).
Phase 0 must first confirm LatentSync 1.6 @ 512 fits this session's VRAM —
run `notebooks/phase0_vram_spike.ipynb` before relying on the 1080p profile.

In [ ]:
# Once the providers exist:
# !talkinghead gen /kaggle/working/script.txt -o /kaggle/working/out/video.mp4 -v

# Until then, the stages that are built can be driven directly:
from talkinghead import media
from talkinghead.config import load_settings
from talkinghead.script_prep import prepare_script

settings = load_settings()
prepared = prepare_script(script_path.read_text(encoding="utf-8"))
print(f"{len(prepared)} segments, ~{prepared.estimated_duration_s():.0f}s")
print(f"work dir: {settings.work_dir}")
print(f"device:   {settings.device}")

## 6. Collect the output

Anything under `/kaggle/working` is downloadable from the session's Output
panel once the notebook finishes.

In [ ]:
!ls -lh /kaggle/working/out/ 2>/dev/null || echo 'no output yet'